# Context Management and RAG Retrieval
Synchronize trusted YAML, inspect indexed records, and test cross-table retrieval.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import sys
ROOT = Path.cwd()
if not (ROOT / 'config').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
from dq_agent.config import load_app_config
from dq_agent.context_store import make_context_retriever
from dq_agent.context_utils import configure_workflow_logging, context_workflow_paths, logged_step

In [ ]:
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
config = load_app_config(ROOT)
paths = context_workflow_paths(config, RUN_ID)
logger = configure_workflow_logging(paths['log'], config.project.log_level)
store = make_context_retriever(config, logger)

In [ ]:
with logged_step(logger, paths['checkpoint'], 'SYNC_CONTEXT'):
    first = store.sync()
    second = store.sync()
    print('First sync:', first)
    print('Idempotent second sync:', second)

In [ ]:
query = {'pair_id': 'sample_fact_sales', 'description': 'sales amount brand relationship', 'columns': ['sales_amount','brand_sid']}
with logged_step(logger, paths['checkpoint'], 'RETRIEVE_CONTEXT', query=query):
    package = store.context_package(query)
display(package['records'])

Verify that exact table context, related relationships, measures, and useful column records appear with provenance and retrieval scores.